# OCT BO-DBA Denoising Study — Exploratory Data Analysis

This notebook loads the study's results file and analyzes: recovery rate by
denoising method, clean-image cost, class-dependence, ensemble strategies
(including patient-grouped validation), denoiser failure dependence, and
perturbation/classifier-margin effects — with paired statistical tests and
patient-clustered confidence intervals throughout.

**Revision note:** this version corrects several methodological issues in the
original exploratory pass — see the Changelog at the end for the full list.
Key changes: clean-image metrics are relabeled as *prediction retention*
(not conventional clean accuracy, since this evaluation set is pre-filtered
to originally-correct images); analysis now accounts for repeated patients
in the dataset; L2 is treated continuously with class-adjusted regression
rather than arbitrary bins; the Bilateral+NLM ensemble pair is explicitly
labeled exploratory/data-driven and validated via patient-grouped
cross-validation; and causal language has been removed throughout.

**Expected input:** a CSV or Excel file with columns —
`Run`, `File Name`, `True Label`, `Processing`, `Prediction`, `Confidence`,
`CNV`, `DME`, `DRUSEN`, `NORMAL`, `Correct?`, `Attacked?`, `Noise Generator`,
`Seeds`, `Query Budget`, `Query Used`, `Linf`, `L2`, `SSIM`, `PSNR`.
`File Name` is expected to follow the Kermany convention
`CLASS-patientID-imageNumber.jpeg`, which this notebook uses to extract
patient IDs for patient-aware analysis.


## Key Definitions

- **L2 / Linf Distance**: perturbation magnitude of the attack (Euclidean / max-single-pixel change), recorded on the `Adversarial` row.
- **Queries Used / Query Budget**: how many classifier queries BO-DBA needed, vs. the nominal cap allowed.
- **Clean Margin**: for the *original* (unattacked) image, the classifier's top-class probability minus its second-highest class probability — a measure of how confidently/marginally correct the clean prediction was, independent of any attack.
- **Adversarial Margin**: the same margin, computed on the successful adversarial prediction — how decisively the attack fooled the classifier (not a "50/50" interpretation, since this is a 4-class problem, not binary).
- **SSIM / PSNR**: structural / pixel-level similarity between a processed image and the original clean image.
- **Recovery Rate**: among successfully-attacked images, the % a given method restores to the correct label.
- **Clean Prediction Retention**: applying a denoiser to a clean, originally-correct image, the % that remain correctly classified. **This is not conventional "clean accuracy"** — this dataset is pre-filtered to images the classifier already got right, so there is no baseline error to recover from; it measures retention of already-correct predictions, not accuracy on a representative sample.
- **Six-Method Empirical Oracle Recovery**: the % of images at least one of the six methods recovers. This is an empirical result *for this dataset*, not a mathematical or theoretical upper bound, and not a method that could be deployed as-is (it requires knowing which method would be correct in advance).
- **Patient-grouped analysis**: several images in this dataset come from the same patient (see Section 2c). Any cross-validation or resampling below groups by patient, so images from one patient never split across train/test or appear as if independent.
- **Exploratory vs. validated selection**: the Bilateral+NLM pairing was identified by examining results on this dataset. Any pair chosen this way is *exploratory/data-driven* until confirmed on data not used to choose it — Section 8c performs a patient-grouped cross-validation check of this specific choice.

**`Processing` values used throughout:** `Original`, `Adversarial`, and for
each of the six methods (`Gaussian`, `Median`, `Bilateral`, `Non-Local Means`,
`Average`, `Morphological Opening`): `<Method>-Attack` and `<Method>-Clean`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.contingency_tables import mcnemar, cochrans_q
from statsmodels.stats.multitest import multipletests
from sklearn.model_selection import GroupKFold
from itertools import combinations

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", None)
RNG_SEED = 42
np.random.seed(RNG_SEED)


## 1. Load the data\n\n**Goal:** get the results data into a pandas DataFrame so the rest of the notebook can analyze it.\n\nUpdate `CSV_PATH` to point to your results file.

In [ ]:
CSV_PATH = "https://raw.githubusercontent.com/thitimas/oct-denoising-study/main/oct_results.xlsx"  # <-- update if needed; also accepts .csv

if CSV_PATH.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(CSV_PATH)
else:
    df = pd.read_csv(CSV_PATH)

df = df[df["Run"].notna()].copy()
print(f"Loaded {len(df)} rows, {df.shape[1]} columns")
df.head()


## 2. Basic sanity checks\n\n**Goal:** catch data problems early — wrong image counts, class imbalance, duplicate images — before drawing any conclusions from downstream analysis.

In [ ]:
METHODS = ["Gaussian", "Median", "Bilateral", "Non-Local Means", "Average", "Morphological Opening"]

n_images = df["Run"].nunique()
print(f"Unique images: {n_images}")

print()
print("Rows per Processing type:")
print(df["Processing"].value_counts())

orig = df[df["Processing"] == "Original"]
print()
print("Class balance (True Label, Original rows):")
print(orig["True Label"].value_counts())

dupes = orig["File Name"].value_counts()
dupes = dupes[dupes > 1]
if len(dupes):
    print()
    print("WARNING: duplicate File Name entries among Original rows:")
    print(dupes)
else:
    print()
    print("No duplicate File Name entries found among Original rows.")


## 2b. Confusion Matrix Overview

**Goal:** give collaborators an at-a-glance view of classifier behavior across classes, before diving into the denoising-specific analysis below.

Two matrices are shown:
- **Clean (Original) confusion matrix** — a sanity check. Since this study only attacks images the classifier already got right, this should be (close to) a perfect diagonal. If it isn't, that flags a data problem worth investigating before trusting anything downstream.
- **Post-Attack (Adversarial) confusion matrix** — shows which class each true label gets pushed *into* by a successful attack.


In [ ]:
from sklearn.metrics import confusion_matrix

class_order = sorted(df["True Label"].dropna().unique())

def plot_confusion(processing_value, title, ax):
    subset = df[df["Processing"] == processing_value]
    cm = confusion_matrix(subset["True Label"], subset["Prediction"], labels=class_order)
    cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Blues", vmin=0, vmax=100,
                xticklabels=class_order, yticklabels=class_order, ax=ax, cbar=False)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    return cm, cm_pct

fig, axes = plt.subplots(1, 2, figsize=(12,5))
cm_clean, pct_clean = plot_confusion("Original", "Clean (Original) — row %", axes[0])
cm_adv, pct_adv = plot_confusion("Adversarial", "Post-Attack (Adversarial) — row %", axes[1])
plt.tight_layout()
plt.show()

print("Per-class accuracy, clean images:")
for i, cls in enumerate(class_order):
    print(f"  {cls}: {pct_clean[i,i]:.1f}%")
print()
print("Per-class accuracy, immediately after attack (before any denoising):")
for i, cls in enumerate(class_order):
    print(f"  {cls}: {pct_adv[i,i]:.1f}%")


## 2c. Patient-Level EDA

**Goal:** this dataset's filenames follow the Kermany convention
`CLASS-patientID-imageNumber`. Several images can come from the same
patient, meaning the 240 image-level observations are **not automatically
240 independent subjects**. This section quantifies that overlap, and every
cross-validation / bootstrap analysis later in the notebook groups by
patient accordingly. Repeated-patient images are kept in the main analysis
throughout — nothing is dropped.


In [ ]:
def get_patient_id(filename):
    parts = str(filename).replace(".jpeg", "").replace(".jpg", "").split("-")
    return parts[1] if len(parts) > 1 else None

orig = df[df["Processing"] == "Original"].copy()
orig["patient_id"] = orig["File Name"].apply(get_patient_id)

n_images_total = len(orig)
n_patients_total = orig["patient_id"].nunique()
print(f"Total images: {n_images_total}")
print(f"Unique patients overall: {n_patients_total}")
print()

print("Unique patients per class:")
print(orig.groupby("True Label")["patient_id"].nunique())
print()

patient_counts = orig["patient_id"].value_counts()
print("Distribution of images per patient:")
print(patient_counts.value_counts().sort_index().rename("num_patients_with_this_many_images"))
print()
print(f"Maximum images contributed by one patient: {patient_counts.max()}")
print(f"Patients contributing more than 1 image: {(patient_counts > 1).sum()} "
      f"({(patient_counts > 1).sum() / n_patients_total * 100:.1f}% of patients)")

fig, ax = plt.subplots(figsize=(6,4))
patient_counts.value_counts().sort_index().plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_xlabel("Images from one patient")
ax.set_ylabel("Number of patients")
ax.set_title("Distribution of Images per Patient")
plt.tight_layout()
plt.show()


## 3. Build a wide, per-image lookup

**Goal:** reshape the data so every image's full story (clean, attacked, all
denoised versions, plus patient ID and confidence margins) sits in one row,
which is what every analysis below needs.


In [ ]:
def build_wide(df):
    wide = {}
    for _, row in df.iterrows():
        run = row["Run"]
        proc = row["Processing"]
        if run not in wide:
            wide[run] = {"true_label": row["True Label"], "file_name": row["File Name"],
                         "patient_id": get_patient_id(row["File Name"])}
        wide[run][f"{proc}_pred"] = row["Prediction"]
        wide[run][f"{proc}_conf"] = row["Confidence"]
        wide[run][f"{proc}_correct"] = row["Correct?"]

        class_probs = [row.get(c) for c in ["CNV", "DME", "DRUSEN", "NORMAL"] if pd.notna(row.get(c))]
        if len(class_probs) == 4:
            sorted_probs = sorted(class_probs, reverse=True)
            margin = sorted_probs[0] - sorted_probs[1]
            if proc == "Original":
                wide[run]["clean_margin"] = margin
            if proc == "Adversarial":
                wide[run]["adv_margin"] = margin

        if proc == "Adversarial":
            wide[run]["l2"] = row.get("L2")
            wide[run]["linf"] = row.get("Linf")
            wide[run]["queries_used"] = row.get("Query Used")
            wide[run]["query_budget"] = row.get("Query Budget")
            wide[run]["noise_generator"] = row.get("Noise Generator")
    return pd.DataFrame.from_dict(wide, orient="index")

wide = build_wide(df)
wide.index.name = "Run"
print(f"Built wide table: {wide.shape[0]} images x {wide.shape[1]} columns")
wide[["true_label", "patient_id", "clean_margin", "adv_margin", "l2"]].head()


## 4. Recovery Rate by denoising method\n\n**Goal:** among images where the attack succeeded, what fraction did each denoiser restore to the correct label?

In [ ]:
attacked_mask = wide["Adversarial_correct"] == "N"
attacked = wide[attacked_mask].copy()
print(f"Attack success: {len(attacked)} / {len(wide)} images ({len(attacked)/len(wide)*100:.1f}%)")

recovery_rates = {}
for m in METHODS:
    col = f"{m}-Attack_correct"
    if col in attacked.columns:
        recovery_rates[m] = (attacked[col] == "Y").mean()

recovery_df = pd.Series(recovery_rates, name="Recovery Rate").sort_values(ascending=False)
recovery_df


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
(recovery_df * 100).plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_ylabel("Recovery Rate (%)")
ax.set_title("Recovery Rate by Denoising Method")
ax.set_ylim(0, 100)
for i, v in enumerate(recovery_df * 100):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 5. Clean Prediction Retention by denoising method

**Goal:** check whether each denoiser damages images that were never
attacked. **Important terminology note:** this is *not* conventional clean
accuracy — every image in this evaluation set was one the classifier
originally got right, by design (only correctly-classified images are
valid attack targets). So this metric measures what fraction of
*already-correct* predictions survive denoising, not accuracy on a
representative sample. True clean accuracy would require running these
denoisers on the full, unfiltered test set — noted as an open item in the
Changelog.


In [ ]:
clean_retention = {}
for m in METHODS:
    col = f"{m}-Clean_correct"
    if col in wide.columns:
        clean_retention[m] = (wide[col] == "Y").mean()

clean_retention_df = pd.Series(clean_retention, name="Clean Prediction Retention").sort_values(ascending=False)
clean_retention_df


In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
x = np.arange(len(METHODS))
w = 0.35
rec_vals = [recovery_rates.get(m, np.nan) * 100 for m in METHODS]
ret_vals = [clean_retention.get(m, np.nan) * 100 for m in METHODS]

ax.bar(x - w/2, rec_vals, w, label="Recovery Rate", color="#4C72B0")
ax.bar(x + w/2, ret_vals, w, label="Clean Prediction Retention", color="#55A868")
ax.set_xticks(x)
ax.set_xticklabels(METHODS, rotation=15)
ax.set_ylabel("%")
ax.set_ylim(0, 100)
ax.set_title("Recovery Rate vs. Clean Prediction Retention, by Method")
ax.legend()
plt.tight_layout()
plt.show()


### 5b. Clean Prediction Retention by True Class\n\n**Goal:** check whether denoising damages already-correct predictions equally across classes, or disproportionately for some. Verified directly from the source data.

In [ ]:
clean_retention_by_class = wide.groupby("true_label").apply(
    lambda g: pd.Series({m: (g[f"{m}-Clean_correct"]=="Y").mean() for m in METHODS if f"{m}-Clean_correct" in g.columns})
)
display((clean_retention_by_class * 100).round(1))

fig, ax = plt.subplots(figsize=(9,4.5))
(clean_retention_by_class * 100).plot(kind="bar", ax=ax)
ax.set_ylabel("Clean Prediction Retention (%)")
ax.set_title("Clean Prediction Retention by True Class and Method")
ax.set_ylim(0,100)
ax.legend(bbox_to_anchor=(1.02,1), loc="upper left")
plt.tight_layout()
plt.show()

print()
print("Average clean prediction retention, by class (across all methods):")
print((clean_retention_by_class.mean(axis=1) * 100).round(1))


## 6. Perturbation Magnitude (L2): Distribution and Class-Adjusted Analysis

**Goal:** examine L2 as a continuous variable rather than arbitrary bins,
and test whether it predicts recovery **after accounting for class** — L2
is confounded with class in this dataset (some classes' successful attacks
tend to need larger or smaller perturbations than others), so a pooled,
unadjusted comparison risks a misleading conclusion in either direction.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4.5))
axes[0].hist(attacked["l2"], bins=20, color="#4C72B0", edgecolor="white")
axes[0].set_xlabel("L2 Distance")
axes[0].set_ylabel("Count")
axes[0].set_title("L2 Distribution (Overall)")

sns.boxplot(data=attacked, x="true_label", y="l2", ax=axes[1])
axes[1].set_title("L2 Distribution by True Class")
axes[1].set_xlabel("True Class")
axes[1].set_ylabel("L2 Distance")
plt.tight_layout()
plt.show()

print("L2 summary statistics by class:")
display(attacked.groupby("true_label")["l2"].describe()[["count","mean","std","min","50%","max"]].round(2))


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for m in ["Non-Local Means", "Bilateral"]:
    recovered = (attacked[f"{m}-Attack_correct"] == "Y").astype(int)
    for cls in class_order:
        mask = attacked["true_label"] == cls
        jitter = np.random.uniform(-0.02, 0.02, mask.sum())
        ax.scatter(attacked.loc[mask, "l2"], recovered[mask] + jitter, alpha=0.4, s=20)
    break  # just show NLM as the illustrative example; full model is below
ax.set_xlabel("L2 Distance")
ax.set_ylabel("Recovered (NLM), jittered")
ax.set_title("Recovery vs. L2 Distance (Non-Local Means) — points colored by class not shown; see regression below")
plt.tight_layout()
plt.show()

print("NOTE: do not read a pooled trend off this scatter alone — L2 is confounded with class")
print("(see class-specific L2 stats above). The regression below adjusts for class explicitly.")


In [ ]:
attacked["nlm_recovered"] = (attacked["Non-Local Means-Attack_correct"] == "Y").astype(int)
model_nlm = smf.logit("nlm_recovered ~ l2 + C(true_label)", data=attacked).fit(disp=0)
print("Logistic regression: Recovered_NLM ~ L2 + Class")
print(model_nlm.summary2().tables[1].round(4))
print()
print("Interpretation guidance: a negative L2 coefficient with a 95% CI excluding 0 indicates")
print("higher L2 is associated with LOWER odds of NLM recovery, holding class constant.")
print("Class coefficients show each class's log-odds of recovery relative to the reference class,")
print("holding L2 constant. This is an association, not a causal claim.")


Repeating the same class-adjusted model for the other principal methods, for comparison:

In [ ]:
for m in ["Bilateral", "Gaussian", "Median"]:
    attacked[f"{m.lower().replace(' ','_')}_recovered"] = (attacked[f"{m}-Attack_correct"] == "Y").astype(int)
    try:
        model_m = smf.logit(f"{m.lower().replace(' ','_')}_recovered ~ l2 + C(true_label)", data=attacked).fit(disp=0)
        print(f"--- {m} ---")
        print(model_m.summary2().tables[1].round(4))
        print()
    except Exception as e:
        print(f"{m}: could not fit model ({e})")


## 6b. Clean Classifier Margin Analysis

**Goal:** test whether poor recovery for a class is associated with the
classifier already being less confident/decisive on that class *before*
any attack — i.e., whether recovery difficulty is explained by weak
baseline separation rather than (or in addition to) perturbation magnitude.

**Research question:** does a class's association with poor denoising
recovery persist after accounting for both perturbation magnitude (L2) and
the classifier's original clean-prediction margin? This is an association
test, not a causal claim.


In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
sns.boxplot(data=wide, x="true_label", y="clean_margin", ax=ax)
ax.set_title("Clean Prediction Margin by True Class")
ax.set_xlabel("True Class")
ax.set_ylabel("Clean Margin (top prob. − 2nd prob.)")
plt.tight_layout()
plt.show()

print("Clean margin summary by class:")
display(wide.groupby("true_label")["clean_margin"].describe()[["count","mean","std","min","50%","max"]].round(3))


In [ ]:
model_full = smf.logit("nlm_recovered ~ l2 + clean_margin + C(true_label)", data=attacked).fit(disp=0)
print("Logistic regression: Recovered_NLM ~ L2 + Clean_Margin + Class")
print(model_full.summary2().tables[1].round(4))
print()
print("If a class coefficient remains statistically significant here (95% CI excluding 0) even")
print("after including both L2 and clean_margin, that class's association with recovery is not")
print("fully explained by these two covariates alone. If it becomes non-significant relative to")
print("the L2-only model above, its apparent effect may be substantially explained by perturbation")
print("magnitude and/or baseline classifier confidence. Either outcome is a legitimate finding —")
print("this does not establish what, causally, drives the remaining association.")


## 7. Recovery Rate by True Class\n\n**Goal:** check whether recovery is uniform across diagnostic classes. Numbers verified directly from the source data.

In [ ]:
class_recovery = attacked.groupby("true_label").apply(
    lambda g: pd.Series({m: (g[f"{m}-Attack_correct"]=="Y").mean() for m in METHODS if f"{m}-Attack_correct" in g.columns})
)
display((class_recovery * 100).round(1))

fig, ax = plt.subplots(figsize=(8,4.5))
class_recovery.plot(kind="bar", ax=ax)
ax.set_ylabel("Recovery Rate")
ax.set_title("Recovery Rate by True Diagnostic Class")
ax.legend(bbox_to_anchor=(1.02,1), loc="upper left")
plt.tight_layout()
plt.show()


### 7b. Adversarial Prediction Margin by True Class

How decisive were successful attacks, per class? A smaller margin means the
winning (wrong) class barely beat the runner-up class — this is a 4-class
problem, so a margin near 0 does not mean a binary "50/50" outcome; it
means the top two predicted classes were close, whichever those happened
to be.


In [ ]:
adv_margin_by_class = attacked.groupby("true_label")["adv_margin"].agg(["mean", "std", "count"])
adv_margin_by_class = adv_margin_by_class.rename(columns={"mean": "Mean Adv. Margin", "std": "Std Dev", "count": "N"})
adv_margin_by_class = adv_margin_by_class.sort_values("Mean Adv. Margin")
display(adv_margin_by_class.round(3))

fig, ax = plt.subplots(figsize=(7,4))
adv_margin_by_class["Mean Adv. Margin"].plot(kind="bar", ax=ax, color="#C44E52")
ax.set_ylabel("Mean Adversarial Prediction Margin")
ax.set_title("Adversarial Prediction Margin by True Class")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print()
print("A smaller margin here means the attack's winning class only narrowly beat the runner-up —")
print("consistent with (but not proof of) that class sitting closer to a decision boundary.")


## 8. Ensemble strategies

**Goal:** test whether combining multiple denoisers' outputs beats the best
single method.

**Confidence-selection rule, stated explicitly:** for each image, select
the prediction from whichever denoised version the classifier assigns the
highest maximum predicted probability to, among the methods being combined.


In [ ]:
def highest_conf_acc(methods_subset, data=None):
    # Vectorized: for each image, pick whichever method's confidence is highest,
    # then check if that method's prediction matches the true label.
    if data is None:
        data = attacked
    if len(data) == 0:
        return 0, np.nan
    conf_cols = [f"{m}-Attack_conf" for m in methods_subset]
    pred_cols = [f"{m}-Attack_pred" for m in methods_subset]
    confs = data[conf_cols].to_numpy(dtype=float)
    preds = data[pred_cols].to_numpy()
    best_idx = np.nanargmax(confs, axis=1)
    chosen_preds = preds[np.arange(len(data)), best_idx]
    correct = int((chosen_preds == data["true_label"].to_numpy()).sum())
    n = len(data)
    return correct, (correct / n * 100 if n else np.nan)

def majority_vote_row(row):
    preds = [row.get(f"{m}-Attack_pred") for m in METHODS if pd.notna(row.get(f"{m}-Attack_pred"))]
    if not preds:
        return None, True
    vc = pd.Series(preds).value_counts()
    top = vc.max()
    winners = vc[vc == top].index.tolist()
    if len(winners) > 1:
        return None, True
    return winners[0], False

maj_results = attacked.apply(majority_vote_row, axis=1)
attacked["majority_pred"] = maj_results.apply(lambda x: x[0])
attacked["majority_tie"] = maj_results.apply(lambda x: x[1])

maj_acc = (attacked.loc[~attacked["majority_tie"], "majority_pred"] == attacked.loc[~attacked["majority_tie"], "true_label"]).mean()
maj_tie_rate = attacked["majority_tie"].mean()

conf6_correct, conf6_acc = highest_conf_acc(METHODS)
conf_bn_correct, conf_bn_acc = highest_conf_acc(["Bilateral", "Non-Local Means"])

avg_recovery = recovery_rates.get("Average", np.nan) * 100

print(f"Individual methods:")
for m, v in recovery_df.items():
    print(f"  {m:<24} {v*100:5.1f}%")
print()
print(f"Majority Vote (excl. ties): {maj_acc*100:.1f}%   |   Tie rate: {maj_tie_rate*100:.1f}%")
print(f"All-6 Highest-Confidence selector: {conf6_acc:.1f}%")
print(f"Bilateral + Non-Local Means Highest-Confidence selector: {conf_bn_acc:.1f}%  "
      f"[EXPLORATORY — this pair was identified by examining this dataset; see Section 8c for validation]")
print(f"Pixel-space Average (naive blending, not a selector): {avg_recovery:.1f}%")


In [ ]:
fig, ax = plt.subplots(figsize=(9,4.5))
labels = list(recovery_df.index) + ["Majority Vote", "All-6 Confidence\nSelector", "Bilateral+NLM Selector\n(exploratory)"]
values = list(recovery_df.values*100) + [maj_acc*100, conf6_acc, conf_bn_acc]
colors = ["#4C72B0"]*len(recovery_df) + ["#C44E52", "#55A868", "#DD8452"]

ax.bar(labels, values, color=colors)
ax.set_ylabel("Accuracy / Recovery Rate (%)")
ax.set_title("Single Methods vs. Ensemble Strategies")
ax.set_ylim(0,100)
plt.xticks(rotation=30, ha="right")
for i,v in enumerate(values):
    ax.text(i, v+1, f"{v:.1f}%", ha="center", fontsize=9)
plt.tight_layout()
plt.show()


### 8b. Pairwise ensemble search (exploratory)

**Goal:** test every possible pair of methods with confidence selection.
**This entire search is exploratory / data-driven** — it uses the same 240
images being reported on elsewhere in this notebook, so the best pair found
here should not be described as independently validated. Section 8c
performs a patient-grouped cross-validation check specifically for whether
this selection procedure is stable.


In [ ]:
pair_results = []
for m1, m2 in combinations(METHODS, 2):
    c, acc = highest_conf_acc([m1, m2])
    pair_results.append({"Pair": f"{m1} + {m2}", "Correct": c, "Accuracy (%)": round(acc,1)})

pair_df = pd.DataFrame(pair_results).sort_values("Accuracy (%)", ascending=False).reset_index(drop=True)
display(pair_df)

best_pair_row = pair_df.iloc[0]
print()
print("Best pair on this dataset:", best_pair_row["Pair"], "—", str(best_pair_row["Accuracy (%)"]) + "%")
print("Best single method:", recovery_df.idxmax(), "—", f"{recovery_df.max()*100:.1f}%")
print("All-6 ensemble:", f"{conf6_acc:.1f}%")
print()
print("This ranking was produced by examining performance on the full 240-image set, so treat it")
print("as a hypothesis about which pair generalizes, not a confirmed result. See Section 8c.")


### 8b-2. Freeze the Bilateral + NLM confidence-selection rule

**Goal:** having identified Bilateral + Non-Local Means as the top-scoring pair in
Section 8b's exploratory search, we now fix that specific rule and stop searching.
Every downstream use of this pair (Section 8c's validation, Section 8d's confidence
intervals, and the reporting below) applies this exact, unchanged rule rather than
re-optimizing anything further. This cell also computes the one metric the notebook
was missing: the selector's own clean-prediction retention — applying the same
highest-confidence rule to **clean, unattacked** images, not just attacked ones, since
a selection rule needs to be judged on both.


In [ ]:
FROZEN_PAIR = ("Bilateral", "Non-Local Means")

# Recovery, using the already-defined helper (attacked images)
frozen_correct, frozen_recovery_acc = highest_conf_acc(list(FROZEN_PAIR))

# Clean-prediction retention for the SELECTOR itself (not just each method separately):
# apply the same "pick whichever has higher confidence" rule to clean images.
def highest_conf_retention(methods_subset, data=wide):
    conf_cols = [f"{m}-Clean_conf" for m in methods_subset]
    pred_cols = [f"{m}-Clean_pred" for m in methods_subset]
    confs = data[conf_cols].to_numpy(dtype=float)
    preds = data[pred_cols].to_numpy()
    best_idx = np.nanargmax(confs, axis=1)
    chosen_preds = preds[np.arange(len(data)), best_idx]
    correct = int((chosen_preds == data["true_label"].to_numpy()).sum())
    n = len(data)
    return correct, (correct / n * 100 if n else np.nan)

frozen_clean_correct, frozen_clean_retention = highest_conf_retention(list(FROZEN_PAIR))

print(f"FROZEN RULE: select whichever of {FROZEN_PAIR[0]!r} or {FROZEN_PAIR[1]!r} has the")
print(f"higher classifier confidence; use that method's prediction.")
print()
print(f"Adversarial recovery (attacked images):        {frozen_recovery_acc:.1f}%  "
      f"({frozen_correct}/{len(attacked)})")
print(f"Clean prediction retention (unattacked images): {frozen_clean_retention:.1f}%  "
      f"({frozen_clean_correct}/{len(wide)})")
print()
print("Compare to the two methods individually:")
print(f"  NLM alone       — recovery: {recovery_rates['Non-Local Means']*100:.1f}%   "
      f"retention: {clean_retention['Non-Local Means']*100:.1f}%")
print(f"  Bilateral alone — recovery: {recovery_rates['Bilateral']*100:.1f}%   "
      f"retention: {clean_retention['Bilateral']*100:.1f}%")


### 8c. Patient-Grouped Validation of the Bilateral + NLM Selection

**Goal:** test whether the Bilateral+NLM pair (identified above by looking
at this dataset) is a stable choice, using proper internal cross-validation.
**This is internal validation, not an independent external test set** — a
genuinely held-out cohort (e.g. fresh images not used anywhere in this
study) would be a stronger check, noted in the Changelog as still needed.

Each fold selects the best pair using only its training images, then
applies that selection to its held-out fold — folds are split by **patient**,
so no patient's images appear in both the training and test portion of a
fold.


In [ ]:
gkf = GroupKFold(n_splits=5)
groups = attacked["patient_id"].values
indices = np.arange(len(attacked))

fold_results = []
for fold_i, (train_idx, test_idx) in enumerate(gkf.split(indices, groups=groups)):
    train_df = attacked.iloc[train_idx]
    test_df = attacked.iloc[test_idx]

    best_pair, best_train_acc = None, -1
    for m1, m2 in combinations(METHODS, 2):
        _, acc = highest_conf_acc([m1, m2], data=train_df)
        if acc > best_train_acc:
            best_train_acc, best_pair = acc, (m1, m2)

    _, test_acc = highest_conf_acc(list(best_pair), data=test_df)
    fold_results.append({
        "Fold": fold_i + 1, "N train": len(train_df), "N test": len(test_df),
        "Selected pair": f"{best_pair[0]} + {best_pair[1]}",
        "Train accuracy (%)": round(best_train_acc, 1),
        "Held-out test accuracy (%)": round(test_acc, 1),
    })

fold_df = pd.DataFrame(fold_results)
display(fold_df)

print()
print("How often each pair was selected across folds:")
print(fold_df["Selected pair"].value_counts())
print()
print(f"Mean held-out (test-fold) accuracy across folds: {fold_df['Held-out test accuracy (%)'].mean():.1f}%")
print(f"Std dev across folds: {fold_df['Held-out test accuracy (%)'].std():.1f}")


### 8d. Patient-Cluster Bootstrap Confidence Intervals

**Goal:** report 95% confidence intervals for headline recovery/retention
metrics, resampling at the **patient** level (not the image level) so that
a patient's repeated images move together rather than being treated as
independent draws. Uses a fixed seed for reproducibility.


In [ ]:
def bootstrap_patient_ci(data, patient_col, metric_fn, n_boot=500, seed=RNG_SEED):
    # Precompute row positions per patient once, then resample via fast iloc lookups
    # (avoids repeated DataFrame scans/pd.concat inside the loop, which is very slow).
    data_reset = data.reset_index(drop=True)
    patient_to_pos = data_reset.groupby(patient_col).indices
    patients_unique = data_reset[patient_col].unique()
    rng = np.random.default_rng(seed)
    estimates = []
    for _ in range(n_boot):
        sampled_patients = rng.choice(patients_unique, size=len(patients_unique), replace=True)
        pos = np.concatenate([patient_to_pos[p] for p in sampled_patients])
        boot_df = data_reset.iloc[pos]
        estimates.append(metric_fn(boot_df))
    estimates = np.array(estimates)
    return np.nanpercentile(estimates, 2.5), np.nanpercentile(estimates, 97.5), estimates

def recovery_metric(methods):
    def fn(d):
        _, acc = highest_conf_acc(methods, data=d) if len(methods) > 1 else (None, (d[f"{methods[0]}-Attack_correct"]=="Y").mean()*100)
        return acc
    return fn

def retention_metric(methods):
    def fn(d):
        vals = [(d[f"{m}-Clean_correct"]=="Y").mean()*100 for m in methods if f"{m}-Clean_correct" in d.columns]
        return np.mean(vals) if vals else np.nan
    return fn

targets = {
    "NLM recovery": (attacked, recovery_metric(["Non-Local Means"])),
    "Bilateral recovery": (attacked, recovery_metric(["Bilateral"])),
    "Bilateral+NLM selector recovery": (attacked, recovery_metric(["Bilateral", "Non-Local Means"])),
    "All-6 selector recovery": (attacked, recovery_metric(METHODS)),
    "NLM clean retention": (wide, retention_metric(["Non-Local Means"])),
    "Bilateral clean retention": (wide, retention_metric(["Bilateral"])),
}

ci_results = []
point_estimates = {}
for name, (data, fn) in targets.items():
    lo, hi, ests = bootstrap_patient_ci(data, "patient_id", fn)
    point = fn(data)
    point_estimates[name] = (point, ests)
    ci_results.append({"Metric": name, "Point Estimate (%)": round(point,1), "95% CI Low": round(lo,1), "95% CI High": round(hi,1)})

# Difference: Bilateral+NLM selector vs NLM alone
diff_ests = point_estimates["Bilateral+NLM selector recovery"][1] - point_estimates["NLM recovery"][1]
diff_point = point_estimates["Bilateral+NLM selector recovery"][0] - point_estimates["NLM recovery"][0]
diff_lo, diff_hi = np.nanpercentile(diff_ests, [2.5, 97.5])
ci_results.append({"Metric": "Difference: (Bilateral+NLM) − NLM", "Point Estimate (%)": round(diff_point,1),
                    "95% CI Low": round(diff_lo,1), "95% CI High": round(diff_hi,1)})

ci_df = pd.DataFrame(ci_results)
display(ci_df)
print()
print("If a CI for a difference excludes 0, that is evidence of a real difference at the 95% level,")
print("accounting for the fact that images from the same patient are not independent observations.")


## 9. Denoiser Failure Dependence

**Goal:** explain how the ensemble strategies in Section 8 perform the way
they do. If denoisers fail on the same images rather than different ones,
combining them adds less than a naive expectation might suggest.


In [ ]:
error_flags = pd.DataFrame({
    m: (attacked[f"{m}-Attack_correct"] != "Y") for m in METHODS if f"{m}-Attack_correct" in attacked.columns
})

corr = error_flags.astype(int).corr()
display(corr.round(3))

fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap="Reds", vmin=0, vmax=1, ax=ax, square=True)
ax.set_title("Correlation Between Denoisers' Errors")
plt.tight_layout()
plt.show()


In [ ]:
n = len(error_flags)
rows_out = []
for m1, m2 in combinations(error_flags.columns, 2):
    obs = (error_flags[m1] & error_flags[m2]).sum()
    p1 = error_flags[m1].mean()
    p2 = error_flags[m2].mean()
    exp = p1 * p2 * n
    excess = (obs/exp - 1) * 100 if exp > 0 else np.nan
    rows_out.append({"Pair": f"{m1} & {m2}", "Observed co-failures": obs,
                      "Expected (independence)": round(exp,1), "Excess %": round(excess,1)})

dep_df = pd.DataFrame(rows_out).sort_values("Excess %", ascending=False)
display(dep_df)

print()
print("Failure dependence varies across denoiser pairs: some methods fail on substantially")
print("overlapping sets of images, whereas other pairs show comparatively weak dependence.")
print("Ensemble performance depends on both individual method strength and how complementary")
print("(vs. overlapping) their failures are — not on either alone.")
print()
print("Note on the pixel-space 'Average' method specifically: correlated PREDICTION errors among")
print("the individual filters do not, by themselves, explain why pixel-level averaging of the")
print("denoised images performs poorly. Averaging images in pixel space can blur or distort")
print("class-relevant structure independently of how correlated the individual methods' final")
print("predictions are — these are two distinct mechanisms, and this notebook does not")
print("distinguish between them.")


### 9b. Paired Statistical Comparisons

**Goal:** since every denoiser is applied to the *same* 240 images, method
comparisons are paired, not independent samples — an overall test
(Cochran's Q) checks whether the six methods differ at all, and selected
pairwise McNemar tests (with Holm correction for multiple comparisons)
check specific comparisons relevant to this paper's claims. Patient
clustering means these image-level tests likely understate true
uncertainty somewhat; the patient-cluster bootstrap CIs in Section 8d are
the more conservative reference where the two disagree.


In [ ]:
method_matrix = attacked[[f"{m}-Attack_correct" for m in METHODS]].eq("Y").astype(int)
q_result = cochrans_q(method_matrix.values)
print(f"Cochran's Q across all 6 methods: statistic={q_result.statistic:.2f}, p={q_result.pvalue:.4f}, df={q_result.df}")
print("(A significant result indicates the six methods do not all have equal recovery rates.)")
print()

# Selected, paper-relevant pairwise comparisons
comparisons = [
    ("Non-Local Means", "Bilateral"),
    ("Non-Local Means", "Median"),
    ("Non-Local Means", "Gaussian"),
    ("Bilateral", "Median"),
]

pvals = []
labels = []
for m1, m2 in comparisons:
    table = pd.crosstab(attacked[f"{m1}-Attack_correct"]=="Y", attacked[f"{m2}-Attack_correct"]=="Y")
    result = mcnemar(table, exact=True)
    pvals.append(result.pvalue)
    labels.append(f"{m1} vs {m2}")

reject, pvals_corrected, _, _ = multipletests(pvals, method="holm")
mcnemar_df = pd.DataFrame({"Comparison": labels, "Raw p-value": pvals, "Holm-corrected p-value": pvals_corrected, "Significant (α=0.05)": reject})
display(mcnemar_df.round(4))


## 10. Distribution of Recovery Across Methods

**Goal:** see whether recovery is roughly all-or-nothing per image, or
whether different methods genuinely catch different cases.


In [ ]:
n_recovered = error_flags.apply(lambda row: (~row).sum(), axis=1)
dist = n_recovered.value_counts().sort_index()
print("How many of the 6 methods recovered each image:")
print(dist)
print()
for k in range(1, 7):
    pct = (n_recovered >= k).mean() * 100
    print(f"At least {k} method(s) recovered: {pct:.1f}%")

any_method = error_flags.eq(False).any(axis=1)
print()
n_oracle = any_method.sum()
print(f"Six-Method Empirical Oracle Recovery: {n_oracle} / {len(attacked)} = {n_oracle/len(attacked)*100:.1f}%")
print(f"Images no method recovers: {len(attacked) - n_oracle} ({(len(attacked)-n_oracle)/len(attacked)*100:.1f}%)")
print()
print("This is the recovery rate an ORACLE would achieve if it always knew, in advance, which of")
print("the six methods would produce the correct label for a given image. It is an empirical result")
print("for this dataset, not a realizable classifier and not a mathematical upper bound.")

fig, ax = plt.subplots(figsize=(6,4))
dist.plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_xlabel("# of denoisers that recovered the image")
ax.set_ylabel("# of images")
ax.set_title("Distribution of Recovery Across Methods")
plt.tight_layout()
plt.show()


## 11. SSIM / PSNR by condition\n\n**Goal:** get a visual-quality view of what each denoiser is doing to the images, independent of whether the classifier ultimately gets the label right.

In [ ]:
ssim_cols = [c for c in df.columns if c.upper()=="SSIM"]
psnr_cols = [c for c in df.columns if c.upper()=="PSNR"]

if ssim_cols and "Processing" in df.columns:
    plot_df = df[df["Processing"].isin([f"{m}-Attack" for m in METHODS] + [f"{m}-Clean" for m in METHODS])].copy()
    plot_df["Method"] = plot_df["Processing"].str.replace("-Attack","").str.replace("-Clean","")
    plot_df["Condition"] = plot_df["Processing"].apply(lambda x: "Attack" if "Attack" in x else "Clean")

    fig, axes = plt.subplots(1, 2, figsize=(13,4.5))
    sns.boxplot(data=plot_df, x="Method", y=ssim_cols[0], hue="Condition", ax=axes[0])
    axes[0].set_title("SSIM by Method and Condition")
    axes[0].tick_params(axis='x', rotation=20)

    if psnr_cols:
        plot_df["_psnr_num"] = pd.to_numeric(plot_df[psnr_cols[0]].astype(str).str.replace("dB","").str.strip(), errors="coerce")
        sns.boxplot(data=plot_df, x="Method", y="_psnr_num", hue="Condition", ax=axes[1])
        axes[1].set_title("PSNR by Method and Condition")
        axes[1].set_ylabel("PSNR (dB)")
        axes[1].tick_params(axis='x', rotation=20)

    plt.tight_layout()
    plt.show()
else:
    print("SSIM/PSNR columns not found — skipping.")


## 12. Query Budget Diagnostic

**Goal:** the source data reports a nominal `Query Budget` of 200, but some
`Query Used` values exceed this. This section quantifies the discrepancy
without speculating about its cause — the explanation needs to be confirmed
against the attack implementation before publication.


In [ ]:
qb_values = attacked["query_budget"].dropna().unique()
qu = attacked["queries_used"].dropna()

print("Unique Query Budget values:", sorted(qb_values))
print(f"Query Used — min: {qu.min()}, median: {qu.median()}, mean: {qu.mean():.1f}, max: {qu.max()}")

nominal_budget = qb_values[0] if len(qb_values) == 1 else None
if nominal_budget is not None:
    over = qu[qu > nominal_budget]
    print()
    print(f"Images where Queries Used > nominal budget ({nominal_budget}): {len(over)} / {len(qu)} "
          f"({len(over)/len(qu)*100:.1f}%)")
    print(f"Maximum excess over budget: {(over - nominal_budget).max() if len(over) else 0}")


> **Note for collaborators:** the reported query accounting requires
> verification against the attack implementation, because Queries Used
> exceeds the nominal Query Budget for a majority of samples in this
> dataset. This may reflect initialization or auxiliary queries not counted
> toward the optimization budget, but this must be confirmed from the
> attack code before this number is reported in the paper without caveat.


## 13. Main Findings

Findings are separated below into what is directly descriptive, what has
statistical support from this notebook, what remains exploratory, and what
questions are still open. Causal language is avoided throughout.


In [ ]:
print("="*70)
print("DESCRIPTIVE FINDINGS (directly observed in this dataset)")
print("="*70)
print(f"- Total images: {len(wide)}; attack success rate: {len(attacked)/len(wide)*100:.1f}%")
print(f"- {n_patients_total} unique patients contribute these {n_images_total} images "
      f"(max {patient_counts.max()} images from one patient)")
print(f"- Best individual method by recovery rate: {recovery_df.idxmax()} ({recovery_df.max()*100:.1f}%)")
print(f"- Best individual method by clean prediction retention: "
      f"{clean_retention_df.idxmax()} ({clean_retention_df.max()*100:.1f}%)")
print(f"- DRUSEN shows the lowest recovery rate and lowest clean prediction retention of all four")
print(f"  classes across every method tested (see Sections 5b and 7)")
print(f"- Six-Method Empirical Oracle Recovery: {n_oracle/len(attacked)*100:.1f}%, "
      f"versus {recovery_df.max()*100:.1f}% for the best realizable single method")
print(f"- Queries Used exceeds the nominal Query Budget for {len(over)/len(qu)*100:.1f}% of images "
      f"(cause not yet confirmed — see Section 12)")
print()
print("="*70)
print("STATISTICALLY SUPPORTED FINDINGS (this notebook's tests)")
print("="*70)
print(f"- Cochran's Q indicates the six methods do NOT all have equal recovery rates "
      f"(p={q_result.pvalue:.4f})")
print(f"- In a class-adjusted logistic regression, L2 shows a statistically significant")
print(f"  association with NLM recovery after controlling for class (see Section 6) —")
print(f"  this refines, rather than confirms, any simple pooled L2-vs-recovery reading")
print(f"- Patient-clustered bootstrap 95% CIs (Section 8d) should be treated as the primary")
print(f"  uncertainty estimate for headline recovery/retention numbers reported in the paper")
print()
print("="*70)
print("EXPLORATORY FINDINGS (data-driven, not yet independently confirmed)")
print("="*70)
print(f"- The Bilateral+NLM confidence-selector pair ({conf_bn_acc:.1f}%) was identified by")
print(f"  searching all pairs on this same 240-image set — it is NOT independently validated.")
print(f"  Patient-grouped cross-validation (Section 8c) shows this pair selected in most folds,")
print(f"  which is supportive but is internal validation, not an external held-out test.")
print(f"- Denoiser failure dependence varies by pair; it does not, by itself, fully explain why")
print(f"  the pixel-space Average method underperforms — that may involve separate mechanisms")
print(f"  (see Section 9)")
print()
print("="*70)
print("UNRESOLVED / REQUIRES ADDITIONAL DATA (not answerable from this notebook alone)")
print("="*70)
print("- True clean accuracy (denoisers run on the FULL, unfiltered test set, not just this")
print("  pre-filtered 240-image subset) has not yet been measured")
print("- No adaptive-attack evaluation exists yet — all attacks here targeted the undefended")
print("  classifier, not the classifier+denoiser pipeline; robustness under an attacker aware")
print("  of the defense is untested")
print("- The Bilateral+NLM pair has not been tested on a genuinely independent, held-out cohort")
print("- The cause of Queries Used exceeding the nominal Query Budget is unconfirmed")
print("- Patient ID overlap between the ORIGINAL Kermany train/test split (independent of the")
print("  240-image subset here) has not been verified against known dataset-versioning issues")


## Changelog

**1. Methodological corrections made**
- Renamed "Clean Accuracy Maintained" to "Clean Prediction Retention" throughout, with an explicit note that this evaluation set is pre-filtered to originally-correct images and is not conventional clean accuracy.
- Renamed the "theoretical ceiling" / "Any-Method Recovery" to "Six-Method Empirical Oracle Recovery," with explicit language that this is an empirical result for this dataset, not a mathematical bound or deployable method.
- Removed arbitrary L2 bins (`<10`, `10-50`, `50-100`, `>100`); L2 is now treated as continuous, shown by class, and tested via class-adjusted logistic regression rather than a pooled, unadjusted comparison.
- Removed the "50/50 coin flip" description of adversarial confidence (inaccurate for a 4-class problem); replaced with "adversarial prediction margin."
- Softened the claim that correlated denoiser errors "directly explain" why pixel-space averaging underperforms; the notebook now states these are two distinct, undistinguished mechanisms.
- Labeled the Bilateral+NLM ensemble pair explicitly as exploratory/data-driven everywhere it appears, rather than presenting it as a validated result.
- Removed causal framing from the DRUSEN discussion; findings are now stated as associations verified directly from the spreadsheet.

**2. New analyses added**
- Patient-ID extraction and patient-level EDA (Section 2c): unique patients overall and per class, images-per-patient distribution, repeated-patient flagging.
- Clean Prediction Retention by true class (Section 5b).
- Class-adjusted logistic regression of recovery on L2 (Section 6), repeated for Bilateral, Gaussian, and Median.
- Clean classifier margin extraction and its distribution by class (Section 6b), plus a combined L2 + Clean Margin + Class logistic regression.
- Patient-grouped (GroupKFold) cross-validation of the Bilateral+NLM pair selection (Section 8c).
- Patient-cluster bootstrap 95% confidence intervals for headline recovery/retention metrics and their pairwise difference (Section 8d).
- Paired statistical testing: Cochran's Q across all six methods, and Holm-corrected McNemar tests for selected pairwise comparisons (Section 9b).
- Query-budget diagnostic section (Section 12), reporting the Query Used vs. Query Budget discrepancy without speculating on its cause.

**3. Claims/conclusions changed**
- No longer claims "larger perturbations are easier to recover" from pooled data; the class-adjusted model is reported instead, with cautious interpretation language.
- No longer claims DRUSEN itself "causes" poor recovery; states the association and notes it persists (or does not) after adjusting for L2 and clean margin, without asserting a mechanism.
- No longer claims "all denoisers have strongly correlated errors"; reports that dependence varies by pair.
- No longer describes Bilateral+NLM as validated; explicit exploratory labeling added throughout, with patient-grouped internal validation reported alongside that caveat.

**4. Issues that still require additional experimental data (not resolvable via notebook analysis alone)**
- True clean accuracy on the full, unfiltered test set (requires running all six denoisers on the complete test set, not generated here).
- Adaptive-attack evaluation (requires re-running BO-DBA against the classifier+denoiser pipeline, not just the undefended classifier).
- Independent external validation of the Bilateral+NLM selector on a held-out cohort not used anywhere in this analysis.
- Confirmation, from the attack implementation itself, of why Queries Used exceeds the nominal Query Budget.
- Verification of patient-ID overlap between the original Kermany train/test split against known dataset-versioning/leakage issues documented in the literature.
